Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). Save to environment as "CEREBRAS_API_KEY".

See README for more details

In [ ]:
SYSTEM_PROMPT_STRING = """
Your task is to partition the 16 words into 4 groups of 4 words based on shared connections. 
Output requirements (STRICT): 
You will be given 16 words.
OUTPUT ONLY the final groups of words.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups.
Make sure there are EXACTLY 4 groups of 4 words each with their category names. NO EXCEPTIONS.
Return the answer exactly in this format:

GROUP 1: word1 || word2 || word3 || word4
GROUP 2: word1 || word2 || word3 || word4
GROUP 3: word1 || word2 || word3 || word4
GROUP 4: word1 || word2 || word3 || word4
"""

USER_PROMPT = "Here are the 16 words: APPLE, PEAR, TABLE, CHAIR, SOFA, BANANA, DESK, PLUM, COUCH, LAMP, ORANGE, STOOL, BED, GRAPE, SHELF, DRAWER"

In [19]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

try:
    response = client.chat.completions.create(
        model="llama3.1-8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_STRING},
            {"role": "user", "content": USER_PROMPT}
        ],
        temperature=0.1,
        max_tokens=600,
    )

    msg = response.choices[0].message
    output = msg.content or ""
    print(output.strip())

except Exception as e:
    print("ERROR:", e)

GROUP 1: APPLE | PEAR | PLUM | GRAPE
GROUP 2: BANANA | ORANGE | TABLE | DESK
GROUP 3: CHAIR | SOFA | COUCH | STOOL
GROUP 4: LAMP | BED | SHELF | DRAWER


In [ ]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = ", ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split("||")] for group in groups ]
    
    return parsed_groups

In [21]:
from data_loader import get_train_test_split

ds_train, ds_test = get_train_test_split()
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

Training puzzles: 521
Testing puzzles: 131


In [25]:
#Number of times LLM can retry if it hallucinates before marking it as an error
MAX_RETRIES = 3

def valid_prediction(pred_groups, words16):
    if pred_groups is None:
        return False
    if len(pred_groups) != 4:
        return False
    if any(len(group) != 4 for group in pred_groups):
        return False
    
    pred_words = [word for group in pred_groups for word in group]
    if set(pred_words) == set(words16):
        return True
    else:
        return False

def solve_puzzle(words16, model="llama3.1-8b", temperature=0.1, max_tokens=600, max_retries=MAX_RETRIES):
    user_prompt = convert_puzzle_to_prompt(words16)

    attempt = 0
    while attempt <= max_retries:
        response = call_llm(
            SYSTEM_PROMPT_STRING,
            user_prompt,
            model=model,
            temperature=(temperature + 0.1*attempt),  
            max_tokens=max_tokens
        )

        pred_groups = parse_response_to_pred_groups(response)

        if valid_prediction(pred_groups, words16):
            return pred_groups

        print(f"Invalid LLM output: {pred_groups}\nRetrying ({attempt+1}/{max_retries})")
        attempt += 1

    print("ERROR: LLM failed after retries")
    return []

In [26]:
row0 = ds_train[0]
words16 = row0["words"]
pred_groups = solve_puzzle(words16)

print("\nPredicted groups:")
for g in pred_groups:
    print(g)

print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])


Predicted groups:
['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
['BOWL', 'LADLE', 'POT', 'SPOON']
['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
['ALIEN', 'AVATAR', 'DUNE', 'TRON']

Gold groups:
MASSIVE -> ['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
USED WHEN SERVING SOUP -> ['BOWL', 'LADLE', 'POT', 'SPOON']
SOMEBODY -> ['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
SCI-FI FRANCHISES -> ['ALIEN', 'AVATAR', 'DUNE', 'TRON']


In [27]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    evaluate,
    gold_groups_from_row,
)

N_EVAL = len(ds_test)
results = evaluate(
    ds_test,
    metric_fns={
        "zero_one": accuracy_zero_one,
        "min_swaps": accuracy_min_swaps,
    },
    solver_fn=solve_puzzle,
    max_samples=N_EVAL,
    gold_from_row=gold_groups_from_row,
    verbose=True
)

acc, n_eval = results["zero_one"]
mean_swaps, _ = results["min_swaps"]
print(f"Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
print(f"Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Processed 10/131 samples
Processed 20/131 samples
Invalid LLM output: [['TEE (GOLF)', 'TEE (SHIRT)', 'GEAR', 'SAW'], ['TEA', 'TEE (SHIRT)', 'ZIPPER', 'COMB'], ['TI (MUSICAL NOTE)', 'GEEZ', 'NUTS', 'RATS'], ['FUDGE', 'BANK', 'BED', 'DELTA']]
Retrying (1/3)
Invalid LLM output: [['TEE (GOLF)', 'TEE (SHIRT)', 'GEAR', 'SAW'], ['TEA', 'TEE (SHIRT)', 'ZIPPER', 'COMB'], ['TI (MUSICAL NOTE)', 'GEEZ', 'NUTS', 'RATS'], ['FUDGE', 'BANK', 'BED', 'DELTA', 'MOUTH']]
Retrying (2/3)
Processed 30/131 samples
Processed 40/131 samples
Processed 50/131 samples
Invalid LLM output: [['CHEERLEADER', 'CHAMPION', 'JOCK', 'HORROR'], ['GOTH', 'METAL', 'PUNK', 'GLAM'], ['ADVOCATE', 'CHAMPION', 'CHEERLEADER', 'VALUE'], ['ABSENT', 'GONE', 'MIA', 'ELSEWHERE']]
Retrying (1/3)
Processed 60/131 samples
Invalid LLM output: []
Retrying (1/3)
Invalid LLM output: []
Retrying (2/3)
Invalid LLM output: []
Retrying (3/3)
Invalid LLM output: []
Retrying (4/3)
ERROR: LLM failed after retries
Invalid LLM output: []
Retrying (1/3)